# RecetIA — Asistente de recetas con RAG en local

Sistema de recomendación y consulta de recetas usando búsqueda semántica con ChromaDB y generación de respuestas en local con Qwen3 (`1.7B`) a través de Ollama.

### Requisitos previos
- Librerías de Python: `pip install datasets pandas sentence-transformers chromadb ollama` (y `flask` para la interfaz web opcional).
- Tener [Ollama](https://ollama.com) instalado y el modelo descargado: `ollama pull qwen3:1.7b`.

## 1. Carga, limpieza y etiquetado exhaustivo

Cargamos el dataset `somosnlp/RecetasDeLaAbuela`, nos quedamos con las columnas relevantes y tomamos una muestra de 400 recetas sin valores nulos ni nombres duplicados.

In [1]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("somosnlp/RecetasDeLaAbuela", "version_1", split="train")
df = ds.to_pandas()

# Seleccionamos las columnas útiles, limpiamos nulos/duplicados y tomamos 400 recetas
df = df[["Nombre", "Ingredientes", "Pasos", "Categoria", "Dificultad", "Valor nutricional"]]
df = df.dropna(subset=["Nombre", "Ingredientes", "Pasos"]).drop_duplicates(subset=["Nombre"])
df = df.sample(400, random_state=42).reset_index(drop=True)

print("Dimensiones:", df.shape)
print(df.isna().sum())
df.head()

Dimensiones: (400, 6)
Nombre                 0
Ingredientes           0
Pasos                  0
Categoria            132
Dificultad           128
Valor nutricional      0
dtype: int64


,Nombre,Ingredientes,Pasos,Categoria,Dificultad,Valor nutricional
0,postre de papaya y yogur,"['½ Papaya', '4 cucharadas soperas de Yogur na...",['1 Reunimos todos los ingredientes para elabo...,"['Recomendada para perder peso', 'Receta sin g...",Dificultad baja,"Alto en calorías, Alto en grasas, Alto en azúcar"
1,sopa de fredolics con albondigas,"['1 cebolla', 'unos cuantos fredolics', '1 lit...","['1 Cortamos los fredolics en trocitos.', '2 E...",NaN,NaN,"Alto en calorías, Alto en grasas, Alto en sodio"
2,carlota de manzana,"['14 galletas de champaña', '3 manzanas hechas...",['1 Cubre el fondo y los costados del molde co...,NaN,NaN,"Alto en calorías, Alto en grasas, Alto en azúcar"
3,guiso de pulpo y patatas,"['400 gramos de Pulpo cocinado', '150 gramos d...",['1 El primer paso para realizar nuestra recet...,[],Dificultad media,"Bajo en calorías, Sin grasa, Alto en fibra"
4,arepas de ocumo chino,"['3 Ocumos chinos grandes (o taro)', '3 cuchar...",['1 ¡Qué fácil es hacer arepas sin harina! En ...,"['Receta sin gluten', 'Recetas venezolanas']",Dificultad baja,"Bajo en calorías, Sin grasa, Alto en fibra"


In [2]:
import re

def limpiar_texto(texto, quitar=()):
    texto = str(texto).lower().translate(str.maketrans("áéíóúü", "aeiouu"))
    for frase in quitar:
        texto = texto.replace(frase, " ")
    return texto

# Frases que parecen alérgenos o ingredientes animales, pero no lo son
EXCLUSIONES = [
    "sin gluten", "sin lactosa", "sin huevo", "sin harina", "sin carne",
    "harina de maiz", "harina de arroz", "harina de garbanzo", "harina de coco", "harina de almendra",
    "leche de soja", "leche de soya", "leche de coco", "leche de almendra", "leche de avena", "leche vegetal", "leche de tigre",
    "manteca de cacao", "manteca vegetal", "tofu de garbanzo", "helado vegano", "queso vegano",
]

ALERGENOS = {
    "gluten": [
        "harina", "pan", "pan rallado", "picatostes", "baguette", "bollo", "hogaza", "sandwich",
        "trigo", "cebada", "centeno", "avena", "cuscus", "couscous", "seitan", "cerveza",
        "pasta", "macarron", "macarrones", "macerrones", "espagueti", "spaghetti", "fideo", "tallarin", "tallarines",
        "fusilli", "tortellini", "lasana", "canelon", "canelones", "pasticho",
        "hojaldre", "masa hojaldrada", "masa brisa", "masa para pizza", "empanada", "empanadilla",
        "galleta", "oreo", "bizcocho", "croissant", "medialuna", "crepe", "pizza", "oblea",
        "salsa de soja", "salsa de soya", "sillao",
    ],
    "lactosa": [
        "leche", "queso", "nata", "mantequilla", "manteqeilla", "manteca", "yogur", "yogures", "yogurt", "yoghurt",
        "kefir", "chantilly", "chantilli", "bechamel", "requeson", "ricota", "ricotta", "mascarpone",
        "mozzarella", "mozarella", "parmesano", "cheddar", "manchego", "cabrales", "roquefort",
        "gouda", "emmental", "emental", "edam", "gruyere", "feta", "pecorino", "cotija", "cottage",
        "philadelphia", "fhiladelfia", "fhiladelphia", "nutella", "toblerone", "helado", "pesto",
        "crema de leche", "crema agria", "crema batida", "crema pastelera", "queso crema", "sopa-crema",
    ],
    "huevo": ["huevo", "clara", "yema", "mayonesa", "alioli", "galletas de champana"],
    "frutos_secos": [
        "frutos secos", "nuez", "nueces", "almendra", "avellana", "cacahuete", "cacahuate",
        "mani", "pistacho", "pinon", "pinones", "anacardo", "pecana", "castana", "macadamia",
        "mazapan", "nutella", "pesto",
    ],
    "pescado": [
        "pescado", "atun", "salmon", "bonito", "bacalao", "merluza", "anchoa", "boqueron", "boquerones",
        "caballa", "sardina", "trucha", "lubina", "dorada", "corvina", "lenguado", "besugo",
        "pargo", "cazon", "tilapia", "mojarra", "pejerrey", "chicharro", "cococha", "kokotxa",
        "surimi", "fume", "fumet", "caldo de pescado", "worcestershire", "salsa inglesa",
    ],
    "marisco": [
        "marisco", "gamba", "gambita", "gambon", "gambones", "camaron", "camarones", "camorones",
        "langostino", "cigala", "quisquilla", "cangrejo", "jaiba", "mejillon", "mejillones",
        "almeja", "chirla", "navaja", "ostra", "vieira", "pulpo", "calamar", "calamares",
        "sepia", "sepieta", "chipiron", "chipirones", "caracol", "caracoles",
    ],
    "soja": ["soja", "soya", "sillao", "tofu", "tempeh", "edamame", "miso"],
    "sesamo": ["sesamo", "ajonjoli", "tahini"],
    "mostaza": ["mostaza", "mustard", "dijon"],
    "apio": ["apio"],
}

ANIMAL = (
    ALERGENOS["lactosa"] + ALERGENOS["huevo"] + ALERGENOS["pescado"] + ALERGENOS["marisco"] + [
        "carne", "carnitas", "pollo", "pollito", "cerdo", "cerde", "puerco", "cochino",
        "lechon", "lechona", "res", "ternera", "vaca", "vacuno", "buey", "cordero", "pato",
        "asado de tira", "cuadril", "jamon", "lacon", "serrano", "iberico", "york",
        "tocino", "beicon", "bacon", "panceta", "chorizo", "choricito", "salchicha", "perritos",
        "longaniza", "morcilla", "sobrasada", "mortadela", "peperoni", "butifarron", "butifarrones",
        "lomo", "lomitos", "solomillo", "solomilloo", "entrecot", "chuleta", "costilla", "ribs", "bistec",
        "pechuga", "muslo", "contramuslo", "albondigas", "higado", "mollejas", "oreja", "pate",
        "codillo", "secreto", "sebo", "manteca de cerdo", "avecrem", "caldo de pollo", "caldo de carne",
        "miel", "gelatina", "colapez", "rana",
    ]
)

def contiene(texto, palabras, extra_excluir=()):
    texto = limpiar_texto(texto, EXCLUSIONES + list(extra_excluir))
    return any(re.search(rf"\b{p}s?\b", texto) for p in palabras)

COLS_ALERGENOS = list(ALERGENOS.keys())
texto_completo = df["Nombre"] + " " + df["Ingredientes"]

for alergeno, palabras in ALERGENOS.items():
    extra = ("manteca de cerdo",) if alergeno == "lactosa" else ()
    if alergeno == "gluten":
        # En recetas keto comprobamos solo ingredientes para evitar falsos positivos con 'pan' en el título
        fuente = df.apply(
            lambda r: r["Ingredientes"] if "keto" in r["Nombre"].lower() else f"{r['Nombre']} {r['Ingredientes']}",
            axis=1,
        )
    else:
        fuente = texto_completo
    df[alergeno] = fuente.apply(lambda t: contiene(t, palabras, extra))

df["tiene_alergenos"] = df[COLS_ALERGENOS].any(axis=1)
df["alergenos"] = df[COLS_ALERGENOS].apply(
    lambda r: ", ".join(c for c in COLS_ALERGENOS if r[c]) or "ninguno",
    axis=1,
)
df["vegano"] = ~texto_completo.apply(lambda t: contiene(t, ANIMAL))

print(df[["vegano", "tiene_alergenos"] + COLS_ALERGENOS].mean().round(4))

vegano             0.1550
tiene_alergenos    0.8150
gluten             0.3750
lactosa            0.4825
huevo              0.2975
frutos_secos       0.1150
pescado            0.1175
marisco            0.1100
soja               0.0475
sesamo             0.0150
mostaza            0.0300
apio               0.0325
dtype: float64


In [3]:
# Revisión sobre 10 recetas al azar mostrando si tienen alérgenos, cuáles y sus ingredientes completos
for _, r in df.sample(10, random_state=1).iterrows():
    print(
        "====", r["Nombre"],
        "| vegano:", r["vegano"],
        "| tiene_alergenos:", r["tiene_alergenos"],
        "| alergenos:", r["alergenos"],
    )
    print(r["Ingredientes"], "\n")

==== pasta con champinones y tocino | vegano: False | tiene_alergenos: True | alergenos: gluten, lactosa
['300 gramos de Pasta corta', '3 lonchas de Tocino o beicon', '100 gramos de Champiñones', '1 pizca de Sal', '1 pizca de Pimienta negra molida', '1 chorro de Aceite de oliva', '100 mililitros de Nata', '½ unidad de Cebolleta'] 

==== kebab de salchicha | vegano: False | tiene_alergenos: True | alergenos: lactosa
['2 Tortillas de maíz para kebab', '4 Salchichas para perritos', '4 Lonchas de queso', '1 chorro de Salsa kebab', '1 puñado de Cebolla frita'] 

==== duraznos con leche condensada | vegano: False | tiene_alergenos: True | alergenos: lactosa
['1 lata de Duraznos en almíbar', '1 chorro de Leche condensada', '1 pizca de Azúcar'] 

==== empanada de carne con cebolla | vegano: False | tiene_alergenos: True | alergenos: gluten, huevo
['2 láminas de hojaldre rectangulares', '700 gramos de carne picada de cerdo y ternera', '1 cebolla', '1 cubo de caldo de carne', '1 vaso pequeño de 

## 2. Embeddings y base vectorial (ChromaDB)

Convertimos cada receta (nombre, categoría e ingredientes) en un vector usando `paraphrase-multilingual-MiniLM-L12-v2` y lo guardamos en ChromaDB (`./chroma_recetas`) junto con todas las etiquetas de alérgenos.

In [4]:
from sentence_transformers import SentenceTransformer
import chromadb

modelo_emb = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# Combinamos nombre, categoría e ingredientes para mejorar la búsqueda semántica
df["texto"] = (
    "Receta: " + df["Nombre"]
    + ". Categoría: " + df["Categoria"].fillna("")
    + ". Ingredientes: " + df["Ingredientes"]
)

embeddings = modelo_emb.encode(df["texto"].tolist(), show_progress_bar=True)

cliente = chromadb.PersistentClient(path="./chroma_recetas")
col = cliente.get_or_create_collection("recetas", metadata={"hnsw:space": "cosine"})

cols_meta = ["Nombre", "vegano", "tiene_alergenos", "alergenos"] + COLS_ALERGENOS
documentos = (
    df["texto"] + "\nAlérgenos detectados: " + df["alergenos"] + "\nPasos: " + df["Pasos"]
).tolist()

# Usamos upsert para poder re-ejecutar esta celda si ajustamos el etiquetado
col.upsert(
    ids=[str(i) for i in df.index],
    embeddings=embeddings.tolist(),
    documents=documentos,
    metadatas=df[cols_meta].to_dict(orient="records"),
)

print("Recetas indexadas en ChromaDB:", col.count())

Recetas indexadas en ChromaDB: 400


## 3. Recuperación semántica y evaluación (Hit@3)

Función de búsqueda sobre ChromaDB que devuelve las `k` recetas más cercanas por distancia coseno (cuanto más baja, más parecida), permitiendo filtrar por metadatos (`vegano`, `tiene_alergenos`, alérgenos concretos, etc.).

In [5]:
def buscar(pregunta, k=3, filtros=None):
    q = modelo_emb.encode([pregunta]).tolist()
    return col.query(query_embeddings=q, n_results=k, where=filtros)

# Prueba rápida con filtro vegano
r = buscar("algo vegano con garbanzos", filtros={"vegano": True})
for m, d in zip(r["metadatas"][0], r["distances"][0]):
    print(round(d, 3), m["Nombre"], "| alérgenos:", m["alergenos"])

0.298 hamburguesa de garbanzos y algarroba | alérgenos: gluten, frutos_secos, soja
0.31 crema de boniato | alérgenos: ninguno
0.327 rollitos de primavera veganos | alérgenos: gluten, soja


### Batería de pruebas (Hit@3)

Evaluamos 10 consultas comprobando si al menos 1 de las 3 recetas recuperadas responde bien a lo pedido, comparando **antes** (búsqueda sin filtros) y **después** (aplicando filtros de alérgenos/vegano):

| # | Pregunta de prueba | Filtro | Top-3 antes (sin filtro) | Acierto (antes) | Top-3 después (con filtro) | Acierto (después) |
|---|---|---|---|---|---|---|
| 1 | Postre sin gluten | `gluten = False` | *gazpacho sin gluten*, *postre de papaya y yogur*, *salsa a la pimienta negra* | Sí | *postre de papaya y yogur*, *salsa a la pimienta negra*, *crema pastelera con maicena* | Sí |
| 2 | Cena rápida con pollo | — | *SECO DE CERDO*, *albondigas a la mallorquina*, *cordero al vapor* | No | *(sin filtro, igual que antes)* | No |
| 3 | Receta vegana con lentejas | `vegano = True` | *hamburguesa de garbanzos y algarroba*, *albondigas de seitan veganas*, *tofu casero paso a paso* | No | *hamburguesa de garbanzos y algarroba*, *rollitos de primavera veganos*, *crema de boniato* | No |
| 4 | Algo con arroz y verduras | — | *arroz con verduras y setas*, *CAZUELA*, *arroz con garbanzos y chorizo* | Sí | *(sin filtro, igual que antes)* | Sí |
| 5 | Sopa para el invierno | — | *sepietas a la plancha*, *Crema fría de espárragos*, *jarabe de anis* | No | *(sin filtro, igual que antes)* | No |
| 6 | Plato sin lactosa con pescado | `lactosa = False` | *langostinos congelados cocidos*, *mbourake senegales*, *rollos de canela y manzana* | No | *langostinos congelados cocidos*, *mbourake senegales*, *tarta de manzana sin gluten y sin lactosa* | No |
| 7 | Bizcocho o tarta sin huevo | `huevo = False` | *huevos con miel*, *hot cakes de avena y platano*, *tarta de queso al horno esponjosa facil y cremosa* | No | *desayuno para ninos monstruo*, *tarta de manzana con crumble*, *cheesecake de oreo sin horno* | Sí |
| 8 | Pasta vegana | `vegano = True` | *espaguetis gratinados al horno*, *hamburguesa de garbanzos y algarroba*, *albondigas de seitan veganas* | No | *hamburguesa de garbanzos y algarroba*, *hummus con salsa tahini*, *rollitos de primavera veganos* | No |
| 9 | Ensalada con aguacate | — | *ensalada de pollo y nueces*, *ensalada fresca con physalis o uchuvas*, *ensalada de pollo y esparragos* | No | *(sin filtro, igual que antes)* | No |
| 10 | Postre sin frutos secos | `frutos_secos = False` | *te de perejil para adelgazar*, *ensalada de pollo paleo*, *pure de frutas para bebes* | No | *te de perejil para adelgazar*, *pure de frutas para bebes*, *zanahoria y perejil* | No |

**Resultado Hit@3:** `2 / 10 = 0.20` (antes) → `3 / 10 = 0.30` (después con filtros).

In [7]:
# Código para comprobar el top-3 antes (sin filtro) y después (con filtro) de las 10 preguntas
pruebas = [
    ("Postre sin gluten", {"gluten": False}),
    ("Cena rápida con pollo", None),
    ("Receta vegana con lentejas", {"vegano": True}),
    ("Algo con arroz y verduras", None),
    ("Sopa para el invierno", None),
    ("Plato sin lactosa con pescado", {"lactosa": False}),
    ("Bizcocho o tarta sin huevo", {"huevo": False}),
    ("Pasta vegana", {"vegano": True}),
    ("Ensalada con aguacate", None),
    ("Postre sin frutos secos", {"frutos_secos": False}),
]

for i, (pregunta, filtro) in enumerate(pruebas, 1):
    antes = [m["Nombre"] for m in buscar(pregunta, k=3)["metadatas"][0]]
    despues = [m["Nombre"] for m in buscar(pregunta, k=3, filtros=filtro)["metadatas"][0]] if filtro else antes
    print(f"{i}. {pregunta}")
    print("   Antes:  ", " | ".join(antes))
    if filtro:
        print("   Después:", " | ".join(despues))

1. Postre sin gluten
   Antes:   gazpacho sin gluten | postre de papaya y yogur | salsa a la pimienta negra
   Después: postre de papaya y yogur | salsa a la pimienta negra | crema pastelera con maicena
2. Cena rápida con pollo
   Antes:   SECO DE CERDO | albondigas a la mallorquina | cordero al vapor
3. Receta vegana con lentejas
   Antes:   hamburguesa de garbanzos y algarroba | albondigas de seitan veganas | tofu casero paso a paso
   Después: hamburguesa de garbanzos y algarroba | rollitos de primavera veganos | crema de boniato
4. Algo con arroz y verduras
   Antes:   arroz con verduras y setas | CAZUELA | arroz con garbanzos y chorizo
5. Sopa para el invierno
   Antes:   sepietas a la plancha | Crema fría de espárragos | jarabe de anis
6. Plato sin lactosa con pescado
   Antes:   langostinos congelados cocidos | mbourake senegales | rollos de canela y manzana
   Después: langostinos congelados cocidos | mbourake senegales | tarta de manzana sin gluten y sin lactosa
7. Bizcocho o 

## 4. Generación de respuestas con Qwen3

Conectamos la búsqueda de ChromaDB con el modelo `qwen3:1.7b` en local mediante Ollama. Le pasamos las 3 recetas recuperadas como contexto para que responda únicamente a partir de ellas y avise de posibles alérgenos.

In [ ]:
import ollama

def asistente(pregunta, filtros=None):
    res = buscar(pregunta, k=3, filtros=filtros)
    contexto = "\n\n---\n\n".join(res["documents"][0])

    # Usamos /no_think para evitar que Qwen3 imprima el bloque largo de razonamiento interno
    prompt = f"""Eres un asistente de cocina. Responde en español usando SOLO las recetas del contexto.
Si ninguna receta encaja, dilo. Indica siempre si la receta tiene alérgenos y cuáles son.

CONTEXTO:
{contexto}

PREGUNTA: {pregunta} /no_think"""

    resp = ollama.chat(model="qwen3:1.7b", messages=[{"role": "user", "content": prompt}])
    return resp["message"]["content"]

print(asistente("Quiero un desayuno", filtros={"vegano": False}))

**Receta: Huevos con miel**  
**Categoría:** No especificada  
**Alérgenos detectados:** gluten, huevo  

**Pasos:**  
1. Tosta una cucharada de miel en un horno a fuego lento.  
2. Añade el huevo batido y mezcla.  
3. Retira del fuego y adóra con picatostes y fruta variada.  

**Nota:** Es una receta sencilla y saludable, ideal para un desayuno rápido.


## 5. Filtros automáticos desde la pregunta

Extraemos directamente de la consulta restricciones como `"vegano"`, `"sin alérgenos"`, `"sin gluten"`, `"sin lactosa"`, `"sin frutos secos"`, etc. para aplicarlas automáticamente en ChromaDB.

In [8]:
PATRONES_FILTRO = {
    "sin gluten": "gluten",
    "sin lactosa": "lactosa",
    "sin huevo": "huevo",
    "sin frutos secos": "frutos_secos",
    "sin pescado": "pescado",
    "sin marisco": "marisco",
    "sin soja": "soja",
    "sin sesamo": "sesamo",
    "sin mostaza": "mostaza",
    "sin apio": "apio",
}

def filtros_de(pregunta):
    p = limpiar_texto(pregunta)
    condiciones = []
    if "vegan" in p:
        condiciones.append({"vegano": True})
    if "sin alergenos" in p:
        condiciones.append({"tiene_alergenos": False})
    for frase, col_alergeno in PATRONES_FILTRO.items():
        if frase in p:
            condiciones.append({col_alergeno: False})
    if not condiciones:
        return None
    return condiciones[0] if len(condiciones) == 1 else {"$and": condiciones}

consulta = "postre vegano sin gluten"
print("Filtros inferidos:", filtros_de(consulta))
print(asistente(consulta, filtros=filtros_de(consulta)))

Filtros inferidos: {'$and': [{'vegano': True}, {'gluten': False}]}
**Receta: crema de boniato**  
**Categoría:** Recomendada para veganos, vegetarianos, perder peso, Receta sin lactosa  
**Alérgenos detectados:** ninguno  

**Pasos:**  
1. Prepara la crema de boniato cocinando el boniato, zanahoria y patata en agua con sal y pimienta.  
2. Calienta agua de caldo, añade las verduras, cocina hasta que estén cocidas.  
3. Triturá las verduras con un batidora eléctrica, añade el agua de caldo y mezcla.  
4. Sirve la crema caliente o fría, con un chorrito de aceite y pimienta.  

**Nota:** Es un postre vegano, sin gluten y sin alérgenos. No incluye gluten ni ingredientes que puedan causar alergias.


## 6. Interfaz web con Flask (opcional)

Pequeño servidor web en local (`http://localhost:5000`) para probar el asistente desde el navegador. Al ejecutar la celda el servidor quedará activo hasta que detengas el kernel.

In [ ]:
from flask import Flask, request

app = Flask(__name__)

HTML = """
<!doctype html>
<meta charset="utf-8">
<title>Asistente de cocina</title>
<h1>Asistente de cocina</h1>
<input id="q" size="50" placeholder="Ej: cena vegana fácil">
<button onclick="preguntar()">Preguntar</button>
<pre id="r" style="white-space: pre-wrap"></pre>
<script>
async function preguntar() {
  r.textContent = "Pensando...";
  const res = await fetch("/preguntar?q=" + encodeURIComponent(q.value));
  r.textContent = await res.text();
}
</script>
"""

@app.route("/")
def inicio():
    return HTML

@app.route("/preguntar")
def preguntar():
    pregunta = request.args.get("q", "")
    return asistente(pregunta, filtros=filtros_de(pregunta))

app.run(port=5000)